In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # BTdiagAI: Complete Implementation
# ## Brain Tumor Classification Using Optimized MRI Preprocessing and Deep Learning Fusion
# ### Paper: Ahmmed et al., Healthcare Technology Letters, 2026
# ### Implements: CLAHE + InceptionV3 + DenseNet121 + Xception + MLP (99.80% accuracy)

# =============================================================================
# SECTION 0: KAGGLE SETUP & LIBRARY INSTALLATION
# =============================================================================

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# Install required packages not in Kaggle by default
install("SimpleITK")
install("imgaug")
install("scikit-image")
install("lightgbm")

# =============================================================================
# SECTION 1: IMPORTS
# =============================================================================

import os
import gc
import json
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from collections import Counter, defaultdict

# Image processing
import cv2
import SimpleITK as sitk
from skimage import exposure, filters, morphology

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception, DenseNet121, InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                         ReduceLROnPlateau, CSVLogger)
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.losses import CategoricalCrossentropy

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import xgboost as xgb
import lightgbm as lgb
import joblib

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# Reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ All imports successful")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

# =============================================================================
# SECTION 2: CONFIGURATION
# =============================================================================

class Config:
    # Paths - Update these to your Kaggle dataset paths
    # Dataset 1: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
    # Dataset 2: https://ieee-dataport.org/documents/brain-mri-nd-5-dataset
    DATASET1_PATH = "/kaggle/input/brain-tumor-mri-dataset"   # Update if needed
    DATASET2_PATH = "/kaggle/input/brain-mri-nd-5-dataset"    # Update if needed
    OUTPUT_PATH   = "/kaggle/working"

    # Image settings (paper uses 224x224)
    IMG_SIZE      = 224
    IMG_SHAPE     = (224, 224, 3)

    # Class names
    CLASSES       = ['glioma', 'meningioma', 'notumor', 'pituitary']
    NUM_CLASSES   = 4

    # Augmentation targets (paper: 2500 per class for Dataset1, 5700 for Dataset2)
    AUG_TARGET_D1 = 2500
    AUG_TARGET_D2 = 5700

    # Training hyperparameters
    BATCH_SIZE    = 32
    EPOCHS        = 50
    LR_ADAM       = 1e-4
    LR_ADAMAX     = 1e-3

    # Feature extraction output dims (from paper)
    XCEPTION_DIM  = 2048
    DENSENET_DIM  = 1024
    INCEPTION_DIM = 2048
    CONCAT_DIM    = 5120   # 2048+1024+2048

    # Feature selection (paper best params)
    ANOVA_K       = 2000
    L1_MAX_FEAT   = 1200

    # CLAHE parameters (paper: clip_limit=0.01, tile_grid=8x8)
    CLAHE_CLIP    = 0.01
    CLAHE_GRID    = (8, 8)

    # Preprocessing method to use for final model
    BEST_PREPROCESS = 'CLAHE'  # Paper's winner

cfg = Config()

# Create output directories
for d in ['models', 'features', 'results', 'plots', 'logs']:
    os.makedirs(os.path.join(cfg.OUTPUT_PATH, d), exist_ok=True)

print(f"\n✅ Config set. Output: {cfg.OUTPUT_PATH}")
print(f"   Classes: {cfg.CLASSES}")
print(f"   Image size: {cfg.IMG_SIZE}x{cfg.IMG_SIZE}")

# =============================================================================
# SECTION 3: DATA LOADING UTILITIES
# =============================================================================

def find_dataset_path(base_path, class_names):
    """Auto-detect dataset folder structure."""
    # Try common Kaggle structures
    candidates = [
        base_path,
        os.path.join(base_path, 'Training'),
        os.path.join(base_path, 'train'),
        os.path.join(base_path, 'data'),
    ]
    for c in candidates:
        if os.path.exists(c):
            subdirs = os.listdir(c)
            matches = sum(1 for s in subdirs if any(cls.lower() in s.lower() for cls in class_names))
            if matches >= 2:
                return c
    return base_path

def load_dataset_paths(dataset_path, class_names, split='all'):
    """
    Load image paths and labels from dataset directory.
    Handles nested train/test/valid splits or flat structure.
    Returns: list of (path, label) tuples
    """
    data = []
    dataset_path = Path(dataset_path)

    # Check for train/test split structure
    has_split = any((dataset_path / s).exists() for s in ['Training', 'Testing', 'train', 'test', 'valid'])

    if has_split:
        splits_to_load = []
        if split == 'all':
            for s in ['Training', 'Testing', 'train', 'test', 'valid', 'Validation']:
                p = dataset_path / s
                if p.exists():
                    splits_to_load.append(p)
        else:
            for s in [split, split.capitalize()]:
                p = dataset_path / s
                if p.exists():
                    splits_to_load.append(p)
                    break
    else:
        splits_to_load = [dataset_path]

    class_map = {}
    for cls in class_names:
        class_map[cls.lower()] = cls
        class_map[cls] = cls

    for split_path in splits_to_load:
        for cls_dir in split_path.iterdir():
            if not cls_dir.is_dir():
                continue
            # Map folder name to class
            folder_lower = cls_dir.name.lower()
            matched_cls = None
            for key, val in class_map.items():
                if key in folder_lower or folder_lower in key:
                    matched_cls = val
                    break
            if matched_cls is None:
                continue
            for img_path in cls_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']:
                    data.append((str(img_path), matched_cls))

    if len(data) == 0:
        # Try flat structure: search recursively
        for cls in class_names:
            for img_path in dataset_path.rglob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    if cls.lower() in str(img_path).lower():
                        data.append((str(img_path), cls))

    print(f"   Found {len(data)} images in {dataset_path}")
    if data:
        dist = Counter([d[1] for d in data])
        for cls, cnt in sorted(dist.items()):
            print(f"     {cls}: {cnt}")
    return data

def load_image(path, size=224):
    """Load and resize image to RGB."""
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Cannot read: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    return img

def load_images_batch(paths, labels, size=224, normalize=True):
    """Load a batch of images into numpy arrays."""
    imgs, lbls = [], []
    for path, label in zip(paths, labels):
        try:
            img = load_image(path, size)
            imgs.append(img)
            lbls.append(label)
        except Exception as e:
            pass
    imgs = np.array(imgs, dtype=np.float32)
    if normalize:
        imgs = imgs / 255.0
    return imgs, lbls

print("✅ Data loading utilities defined")

# =============================================================================
# SECTION 4: PREPROCESSING TECHNIQUES (All 5 from paper)
# =============================================================================

class MRIPreprocessor:
    """
    Implements all 5 preprocessing methods from the paper:
    1. CLAHE (winner: 99.80%)
    2. Nyul Normalization
    3. White Stripe Normalization
    4. N4 Bias Correction
    5. Template Registration
    Plus: Min-Max Normalization (applied to all)
    """

    def __init__(self, config):
        self.cfg = config

    # -------------------------------------------------------------------------
    # 1. Min-Max Normalization (eq. 1 in paper) - applied to ALL methods
    # -------------------------------------------------------------------------
    def minmax_normalize(self, img):
        """I' = (I - I_min) / (I_max - I_min)"""
        img = img.astype(np.float32)
        i_min, i_max = img.min(), img.max()
        if i_max - i_min < 1e-8:
            return np.zeros_like(img)
        return (img - i_min) / (i_max - i_min)

    # -------------------------------------------------------------------------
    # 2. CLAHE (paper best method: clip_limit=0.01, tile_grid=8x8)
    # Equations 2-7 in paper
    # -------------------------------------------------------------------------
    def apply_clahe(self, img):
        """
        Contrast Limited Adaptive Histogram Equalization.
        clip_limit=0.01, tileGridSize=(8,8) per paper Section 3.2.3.2
        """
        # Convert to grayscale-like processing on each channel
        img_uint8 = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        clahe = cv2.createCLAHE(
            clipLimit=self.cfg.CLAHE_CLIP * 255,  # OpenCV uses absolute clip
            tileGridSize=self.cfg.CLAHE_GRID
        )
        # Apply CLAHE to L channel in LAB space (better for MRI)
        if img_uint8.ndim == 3 and img_uint8.shape[2] == 3:
            lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            l_clahe = clahe.apply(l)
            lab_clahe = cv2.merge([l_clahe, a, b])
            result = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
        else:
            result = clahe.apply(img_uint8)

        result = result.astype(np.float32) / 255.0
        # Apply min-max (paper applies it to all)
        return self.minmax_normalize(result)

    # -------------------------------------------------------------------------
    # 3. Nyul Normalization (equations 8-13 in paper)
    # -------------------------------------------------------------------------
    def apply_nyul(self, img, num_landmarks=15, percentile_range=(5, 95)):
        """
        Nyul histogram normalization.
        Standardizes intensity values via percentile landmark matching.
        """
        img_f = img.astype(np.float32)

        # Step 1: Brain region segmentation (eq. 8)
        gray = cv2.cvtColor((img_f * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY) \
               if img_f.ndim == 3 else img_f
        alpha = 0.1
        T = alpha * gray.max()
        mask = (gray > T).astype(np.uint8)
        # Morphological closing with 3x3 kernel
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        brain_pixels = gray[mask > 0]
        if len(brain_pixels) < 10:
            return self.minmax_normalize(img_f)

        # Step 2: Compute reference landmarks (eq. 9-10)
        epsilon = 1e-8
        i_norm = (brain_pixels - brain_pixels.min()) / \
                 (brain_pixels.max() - brain_pixels.min() + epsilon)
        percentiles = np.linspace(percentile_range[0], percentile_range[1], num_landmarks)
        landmarks = np.percentile(i_norm, percentiles)
        # Add boundary points [0, 1]
        ref_landmarks = np.concatenate([[0], landmarks, [1]])
        ref_percentiles = np.concatenate([[0], percentiles, [100]])

        # Step 3: Compute image-specific landmarks (eq. 11)
        img_gray = gray.astype(np.float32)
        ig_norm = (img_gray - img_gray.min()) / (img_gray.max() - img_gray.min() + epsilon)
        img_landmarks = np.percentile(ig_norm.flatten(), ref_percentiles)

        # Step 4: Piecewise linear intensity mapping (eq. 12)
        mapped = np.interp(ig_norm.flatten(), img_landmarks, ref_landmarks)
        mapped = mapped.reshape(img_gray.shape)

        # Step 5: Rescale and preserve background (eq. 13)
        result = np.where(mask > 0, mapped * 255, 0).astype(np.float32) / 255.0

        # Convert back to 3-channel
        if img.ndim == 3:
            result_3ch = np.stack([result] * 3, axis=-1)
        else:
            result_3ch = result

        return self.minmax_normalize(result_3ch)

    # -------------------------------------------------------------------------
    # 4. White Stripe Normalization (equations 14-17 in paper)
    # -------------------------------------------------------------------------
    def apply_white_stripe(self, img, bandwidth=20.0, tau=0.15):
        """
        White Stripe normalization via Z-score using white stripe statistics.
        """
        img_f = img.astype(np.float32)
        gray = cv2.cvtColor((np.clip(img_f, 0, 1) * 255).astype(np.uint8),
                             cv2.COLOR_RGB2GRAY).astype(np.float32)

        # Step 1: Brain mask using Otsu (eq. 14)
        _, otsu_thresh = cv2.threshold(gray.astype(np.uint8), 0, 255,
                                        cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mask = (gray > 0.8 * otsu_thresh).astype(np.uint8)
        if mask.sum() < 100:
            mask = (gray > gray.mean()).astype(np.uint8)

        brain_pixels = gray[mask > 0]
        if len(brain_pixels) < 10:
            return self.minmax_normalize(img_f)

        # Step 2: White stripe detection (eq. 15)
        p2, p50, p98 = np.percentile(brain_pixels, [2, 50, 98])
        # Smooth histogram and find peak
        hist, bin_edges = np.histogram(brain_pixels, bins=256, density=True)
        from scipy.ndimage import gaussian_filter1d
        smooth_hist = gaussian_filter1d(hist, sigma=bandwidth / (p98 - p2 + 1e-8) * 10)
        peak_bin = np.argmax(smooth_hist)
        I_peak = (bin_edges[peak_bin] + bin_edges[peak_bin + 1]) / 2.0

        # Select intensities within ±15% of peak (eq. 15)
        ws_mask = (brain_pixels >= 0.85 * I_peak) & (brain_pixels <= 1.15 * I_peak)
        ws_pixels = brain_pixels[ws_mask]

        if len(ws_pixels) < 5:
            ws_pixels = brain_pixels

        # Step 3: Z-score normalization (eq. 16)
        mu_ws = ws_pixels.mean()
        sigma_ws = ws_pixels.std()

        if sigma_ws < 1e-8:
            result = gray / (gray.max() + 1e-8)
        else:
            result = (gray - mu_ws) / sigma_ws

        # Step 4: Final scaling (eq. 17): preserve 5-95% range
        p5, p95 = np.percentile(result[mask > 0], [5, 95])
        denom = (p95 - p5)
        if denom < 1e-8:
            denom = 1.0
        result = (result - p5) / denom
        # Background = 0
        result = np.where(mask > 0, result, 0)

        result_3ch = np.stack([result] * 3, axis=-1) if img.ndim == 3 else result
        return self.minmax_normalize(result_3ch)

    # -------------------------------------------------------------------------
    # 5. N4 Bias Correction (equation 18 in paper) via SimpleITK
    # -------------------------------------------------------------------------
    def apply_n4_bias_correction(self, img):
        """
        N4ITK bias correction via SimpleITK (eq. 18: I_corrected = I_orig / B(x))
        """
        try:
            img_f = img.astype(np.float32)
            gray = cv2.cvtColor((np.clip(img_f, 0, 1) * 255).astype(np.uint8),
                                 cv2.COLOR_RGB2GRAY).astype(np.float32)

            # Convert to SimpleITK image
            sitk_img = sitk.GetImageFromArray(gray)
            sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32)

            # Create brain mask
            mask_arr = (gray > gray.max() * 0.1).astype(np.uint8)
            sitk_mask = sitk.GetImageFromArray(mask_arr)
            sitk_mask = sitk.Cast(sitk_mask, sitk.sitkUInt8)

            # N4 bias field correction
            corrector = sitk.N4BiasFieldCorrectionImageFilter()
            corrector.SetMaximumNumberOfIterations([50, 40, 30])
            corrected = corrector.Execute(sitk_img, sitk_mask)

            corrected_arr = sitk.GetArrayFromImage(corrected).astype(np.float32)
            corrected_arr = np.clip(corrected_arr, 0, None)

            result_3ch = np.stack([corrected_arr] * 3, axis=-1) if img.ndim == 3 else corrected_arr
            return self.minmax_normalize(result_3ch)

        except Exception as e:
            # Fallback to original if N4 fails
            return self.minmax_normalize(img.astype(np.float32))

    # -------------------------------------------------------------------------
    # 6. Template Registration (equations 19-21 in paper)
    # Using SimpleITK Euler2DTransform
    # -------------------------------------------------------------------------
    def apply_template_registration(self, img, template=None):
        """
        Template-based image registration using Euler2D transform.
        Aligns image to MNI152 template-like reference (eq. 19-21).
        """
        try:
            img_f = img.astype(np.float32)
            gray = cv2.cvtColor((np.clip(img_f, 0, 1) * 255).astype(np.uint8),
                                 cv2.COLOR_RGB2GRAY).astype(np.float32)

            if template is None:
                # Create a synthetic template (MNI152-like: oval brain mask)
                H, W = gray.shape
                template_arr = np.zeros((H, W), dtype=np.float32)
                cy, cx = H // 2, W // 2
                ry, rx = int(H * 0.4), int(W * 0.4)
                Y, X = np.ogrid[:H, :W]
                brain_mask = ((X - cx)**2 / rx**2 + (Y - cy)**2 / ry**2) <= 1
                template_arr[brain_mask] = 128.0
                # Add gradient to simulate MRI-like template
                template_arr += (Y / H * 20).astype(np.float32)
                template_arr = np.clip(template_arr, 0, 255)
            else:
                template_arr = template.astype(np.float32)

            # Convert to SimpleITK
            fixed_img  = sitk.GetImageFromArray(template_arr)
            moving_img = sitk.GetImageFromArray(gray)

            # Registration: Euler2D (eq. 19)
            # shrink_factors=[2,1], smoothing_sigmas=[1,0], max_iter=100
            registration = sitk.ImageRegistrationMethod()
            registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
            registration.SetOptimizerAsGradientDescent(
                learningRate=0.1,
                numberOfIterations=100,
                convergenceMinimumValue=1e-6,
                convergenceWindowSize=10
            )
            registration.SetOptimizerScalesFromPhysicalShift()

            tx = sitk.CenteredTransformInitializer(
                fixed_img, moving_img,
                sitk.Euler2DTransform(),
                sitk.CenteredTransformInitializerFilter.GEOMETRY
            )
            registration.SetInitialTransform(tx, inPlace=False)
            registration.SetInterpolator(sitk.sitkBSpline)

            # Multi-resolution (eq. 20)
            registration.SetShrinkFactorsPerLevel(shrinkFactors=[2, 1])
            registration.SetSmoothingSigmasPerLevel(smoothingSigmas=[1, 0])
            registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

            try:
                final_tx = registration.Execute(
                    sitk.Cast(fixed_img, sitk.sitkFloat32),
                    sitk.Cast(moving_img, sitk.sitkFloat32)
                )
                # B-spline interpolation resampling (eq. 21)
                resampled = sitk.Resample(
                    moving_img, fixed_img, final_tx,
                    sitk.sitkBSpline, 0.0, moving_img.GetPixelID()
                )
                result_arr = sitk.GetArrayFromImage(resampled).astype(np.float32)
            except:
                result_arr = gray

            result_3ch = np.stack([result_arr] * 3, axis=-1) if img.ndim == 3 else result_arr
            return self.minmax_normalize(result_3ch)

        except Exception as e:
            return self.minmax_normalize(img.astype(np.float32))

    # -------------------------------------------------------------------------
    # Main preprocessing dispatcher
    # -------------------------------------------------------------------------
    def preprocess(self, img, method='CLAHE'):
        """
        Apply preprocessing method to image.
        img: numpy array [H, W, 3] float32 in [0,1]
        method: 'CLAHE' | 'Nyul' | 'WhiteStripe' | 'N4' | 'Template'
        """
        # Always resize first
        if img.shape[:2] != (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE):
            img = cv2.resize(img, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))

        img = img.astype(np.float32)
        if img.max() > 1.0:
            img = img / 255.0

        if method == 'CLAHE':
            return self.apply_clahe(img)
        elif method == 'Nyul':
            return self.apply_nyul(img)
        elif method == 'WhiteStripe':
            return self.apply_white_stripe(img)
        elif method == 'N4':
            return self.apply_n4_bias_correction(img)
        elif method == 'Template':
            return self.apply_template_registration(img)
        else:
            return self.minmax_normalize(img)

    def preprocess_batch(self, image_paths, method='CLAHE', desc='Preprocessing'):
        """Preprocess a list of image paths."""
        results = []
        for path in tqdm(image_paths, desc=f'{desc} [{method}]'):
            try:
                img = load_image(path, self.cfg.IMG_SIZE)
                img = img.astype(np.float32) / 255.0
                processed = self.preprocess(img, method)
                results.append(processed)
            except Exception as e:
                # Return zeros on failure
                results.append(np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3),
                                        dtype=np.float32))
        return np.array(results, dtype=np.float32)

preprocessor = MRIPreprocessor(cfg)
print("✅ All 5 preprocessing methods implemented")

# =============================================================================
# SECTION 5: DATA AUGMENTATION (paper Section 3.2.2)
# =============================================================================

class DataAugmentor:
    """
    Implements augmentation pipeline from paper (Section 3.2.2):
    - Rotate, Crop, Sharpen, Contrast Normalization
    - Shear, Gaussian Blur, Gaussian Noise, Zoom
    Balances dataset to target_per_class samples.
    """

    def __init__(self, config):
        self.cfg = config

    def augment_image(self, img):
        """Apply random augmentations to a single image."""
        img = img.astype(np.float32)
        if img.max() > 1.0:
            img = img / 255.0

        augmentations = [
            self._rotate,
            self._crop_zoom,
            self._sharpen,
            self._contrast_normalize,
            self._shear,
            self._gaussian_blur,
            self._gaussian_noise,
            self._zoom,
            self._horizontal_flip,
        ]

        # Apply 1-3 random augmentations
        num_augs = random.randint(1, 3)
        chosen = random.sample(augmentations, num_augs)
        for aug_fn in chosen:
            img = aug_fn(img)
            img = np.clip(img, 0, 1)

        return img

    def _rotate(self, img):
        angle = random.uniform(-20, 20)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    def _crop_zoom(self, img):
        scale = random.uniform(0.85, 1.0)
        h, w = img.shape[:2]
        crop_h, crop_w = int(h * scale), int(w * scale)
        top  = random.randint(0, h - crop_h)
        left = random.randint(0, w - crop_w)
        cropped = img[top:top+crop_h, left:left+crop_w]
        return cv2.resize(cropped, (w, h))

    def _sharpen(self, img):
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
        img_uint8 = (img * 255).astype(np.uint8)
        sharp = cv2.filter2D(img_uint8, -1, kernel)
        return sharp.astype(np.float32) / 255.0

    def _contrast_normalize(self, img):
        factor = random.uniform(0.8, 1.2)
        mean   = img.mean()
        return np.clip((img - mean) * factor + mean, 0, 1)

    def _shear(self, img):
        shear = random.uniform(-0.2, 0.2)
        h, w  = img.shape[:2]
        M = np.float32([[1, shear, 0], [0, 1, 0]])
        return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    def _gaussian_blur(self, img):
        ksize = random.choice([3, 5])
        sigma = random.uniform(0.5, 1.5)
        img_uint8 = (img * 255).astype(np.uint8)
        blurred = cv2.GaussianBlur(img_uint8, (ksize, ksize), sigma)
        return blurred.astype(np.float32) / 255.0

    def _gaussian_noise(self, img):
        noise = np.random.normal(0, random.uniform(0.01, 0.05), img.shape).astype(np.float32)
        return np.clip(img + noise, 0, 1)

    def _zoom(self, img):
        factor = random.uniform(1.0, 1.15)
        h, w   = img.shape[:2]
        new_h, new_w = int(h * factor), int(w * factor)
        zoomed = cv2.resize(img, (new_w, new_h))
        start_h = (new_h - h) // 2
        start_w = (new_w - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]

    def _horizontal_flip(self, img):
        return cv2.flip(img, 1)

    def balance_dataset(self, images, labels, target_per_class):
        """
        Balance dataset by augmenting under-represented classes.
        Returns balanced images and labels arrays.
        """
        class_indices = defaultdict(list)
        for i, lbl in enumerate(labels):
            class_indices[lbl].append(i)

        balanced_images, balanced_labels = [], []

        for cls, indices in class_indices.items():
            cls_imgs = [images[i] for i in indices]
            n_existing = len(cls_imgs)

            # Add original images
            balanced_images.extend(cls_imgs)
            balanced_labels.extend([cls] * n_existing)

            # Generate augmented images if needed
            n_needed = target_per_class - n_existing
            if n_needed > 0:
                for _ in range(n_needed):
                    src_img = random.choice(cls_imgs)
                    aug_img = self.augment_image(src_img)
                    balanced_images.append(aug_img)
                    balanced_labels.append(cls)

        # Shuffle
        combined = list(zip(balanced_images, balanced_labels))
        random.shuffle(combined)
        balanced_images, balanced_labels = zip(*combined)

        return list(balanced_images), list(balanced_labels)

augmentor = DataAugmentor(cfg)
print("✅ Data augmentation pipeline implemented")

# =============================================================================
# SECTION 6: LOAD DATASETS
# =============================================================================

print("\n" + "="*60)
print("SECTION 6: LOADING DATASETS")
print("="*60)

def load_dataset_safe(path, class_names, dataset_name="Dataset"):
    """Load dataset with fallback if path doesn't exist."""
    if not os.path.exists(path):
        print(f"⚠️  {dataset_name} path not found: {path}")
        print(f"   Creating synthetic dataset for demonstration...")
        return create_synthetic_dataset(class_names, n_per_class=50)

    data = load_dataset_paths(path, class_names)
    if len(data) == 0:
        print(f"⚠️  No images found at {path}")
        return create_synthetic_dataset(class_names, n_per_class=50)

    paths  = [d[0] for d in data]
    labels = [d[1] for d in data]
    return paths, labels

def create_synthetic_dataset(class_names, n_per_class=50):
    """Create synthetic MRI-like dataset for testing when real data unavailable."""
    print(f"   Generating synthetic MRI images ({n_per_class} per class)...")
    img_dir = os.path.join(cfg.OUTPUT_PATH, 'synthetic_data')

    all_paths, all_labels = [], []
    for cls_idx, cls in enumerate(class_names):
        cls_dir = os.path.join(img_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)
        for i in range(n_per_class):
            # Generate synthetic brain-like MRI image
            img = np.random.randint(30, 200, (224, 224, 3), dtype=np.uint8)
            # Add oval brain-like shape
            mask = np.zeros((224, 224), dtype=np.uint8)
            cv2.ellipse(mask, (112, 112), (85, 100), 0, 0, 360, 255, -1)
            for c in range(3):
                img[:, :, c] = np.where(mask > 0, img[:, :, c], 20)
            # Add class-specific "tumor" region
            if cls != 'notumor':
                cx = 80 + cls_idx * 20
                cy = 100 + cls_idx * 15
                radius = 20 + cls_idx * 5
                cv2.circle(img, (cx, cy), radius, (180 + cls_idx*15,) * 3, -1)

            path = os.path.join(cls_dir, f'{cls}_{i:04d}.jpg')
            cv2.imwrite(path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            all_paths.append(path)
            all_labels.append(cls)

    print(f"   ✅ Synthetic dataset: {len(all_paths)} images")
    return all_paths, all_labels

# Load Dataset 1
print("\n📂 Loading Dataset 1 (7023 images expected)...")
d1_paths, d1_labels = load_dataset_safe(cfg.DATASET1_PATH, cfg.CLASSES, "Dataset 1")
print(f"   Total: {len(d1_paths)} images")

# Load Dataset 2 (optional - comment out if not available)
print("\n📂 Loading Dataset 2 (17888 images expected)...")
USE_DATASET2 = os.path.exists(cfg.DATASET2_PATH)
if USE_DATASET2:
    d2_paths, d2_labels = load_dataset_safe(cfg.DATASET2_PATH, cfg.CLASSES, "Dataset 2")
    print(f"   Total: {len(d2_paths)} images")
else:
    print("   Dataset 2 not found. Will use Dataset 1 only.")
    d2_paths, d2_labels = d1_paths.copy(), d1_labels.copy()

# =============================================================================
# SECTION 7: VISUALIZE SAMPLE IMAGES
# =============================================================================

print("\n" + "="*60)
print("SECTION 7: VISUALIZATION")
print("="*60)

def plot_sample_images(paths, labels, class_names, title="Sample MRI Images", n_per_class=2):
    """Plot sample images for each class."""
    fig, axes = plt.subplots(len(class_names), n_per_class,
                              figsize=(n_per_class * 3, len(class_names) * 3))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    class_paths = defaultdict(list)
    for p, l in zip(paths, labels):
        class_paths[l].append(p)

    for row, cls in enumerate(class_names):
        cls_imgs = class_paths[cls][:n_per_class]
        for col in range(n_per_class):
            ax = axes[row][col] if n_per_class > 1 else axes[row]
            if col < len(cls_imgs):
                try:
                    img = load_image(cls_imgs[col])
                    ax.imshow(img)
                except:
                    ax.imshow(np.zeros((224, 224, 3)))
            ax.set_title(f'{cls}' if col == 0 else '', fontsize=9)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', 'sample_images.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

def plot_class_distribution(labels, title="Class Distribution"):
    """Plot class distribution pie chart (like paper Figure 2)."""
    counter = Counter(labels)
    classes = list(counter.keys())
    counts  = list(counter.values())
    total   = sum(counts)

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        counts, labels=classes, colors=colors[:len(classes)],
        autopct=lambda pct: f'{pct:.1f}%\n({int(round(pct/100*total))})',
        startangle=90, pctdistance=0.75
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', f'{title.replace(" ", "_")}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

# Dataset 1 distribution
plot_class_distribution(d1_labels, "Dataset 1 Class Distribution")

# Sample images
plot_sample_images(d1_paths, d1_labels, cfg.CLASSES,
                   "Dataset 1: Sample MRI Images")

# Visualize preprocessing methods on one image
def visualize_preprocessing(image_path, preprocessor):
    """Compare all 5 preprocessing methods (like paper Figure 6-10)."""
    methods = ['CLAHE', 'Nyul', 'WhiteStripe', 'N4', 'Template']
    labels_list = ['Original', 'CLAHE', 'Nyul', 'White Stripe', 'N4 Bias', 'Template Reg.']

    try:
        orig = load_image(image_path, cfg.IMG_SIZE)
        orig_f = orig.astype(np.float32) / 255.0

        results = [orig]
        for method in methods:
            processed = preprocessor.preprocess(orig_f.copy(), method)
            results.append((processed * 255).astype(np.uint8))

        fig, axes = plt.subplots(1, 6, figsize=(20, 4))
        fig.suptitle('MRI Preprocessing Methods Comparison', fontsize=14, fontweight='bold')

        for ax, img, lbl in zip(axes, results, labels_list):
            ax.imshow(img if img.dtype == np.uint8 else img)
            ax.set_title(lbl, fontsize=9, fontweight='bold')
            ax.axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', 'preprocessing_comparison.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        print("✅ Preprocessing visualization saved")
    except Exception as e:
        print(f"⚠️  Preprocessing viz failed: {e}")

if d1_paths:
    visualize_preprocessing(d1_paths[0], preprocessor)

print("\n✅ Dataset loading and visualization complete")
print(f"   Dataset 1: {len(d1_paths)} images")
print(f"   Dataset 2: {len(d2_paths)} images")
# =============================================================================
# SECTION 8: DEEP LEARNING FEATURE EXTRACTORS (Paper Section 3.3)
# InceptionV3 + DenseNet121 + Xception → Concatenated 5120-dim features
# =============================================================================

print("\n" + "="*60)
print("SECTION 8: BUILDING DEEP LEARNING FEATURE EXTRACTORS")
print("="*60)

def build_xception(input_shape, num_classes=4):
    """
    Xception model (paper Section 3.3.1):
    - 36 conv layers: entry/middle/exit flow
    - Flatten → Dropout(0.3) → Dense(128, relu) → Dropout(0.25) → Dense(4, softmax)
    - Optimizer: Adamax
    """
    base = Xception(weights='imagenet', include_top=False,
                    input_shape=input_shape)

    x = base.output
    x = layers.Flatten()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.25)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output, name='Xception_FineTuned')

    # Freeze early layers, train final layers
    for layer in base.layers[:-20]:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True

    model.compile(
        optimizer=Adamax(learning_rate=cfg.LR_ADAMAX),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_densenet121(input_shape, num_classes=4):
    """
    DenseNet121 model (paper Section 3.3.2):
    - 121 layers, 4 dense blocks
    - GlobalAveragePooling → Dropout(0.4) → Dense(256, relu) → Dropout(0.3) → Dense(4, softmax)
    - Optimizer: Adam
    """
    base = DenseNet121(weights='imagenet', include_top=False,
                       input_shape=input_shape)

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output, name='DenseNet121_FineTuned')

    for layer in base.layers[:-20]:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True

    model.compile(
        optimizer=Adam(learning_rate=cfg.LR_ADAM),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_inceptionv3(input_shape, num_classes=4):
    """
    InceptionV3 model (paper Section 3.3.3):
    - Factorized convolutions (3×3 → 3×1 and 1×3)
    - GlobalAveragePooling → Dropout(0.4) → Dense(256, relu) → Dropout(0.3) → Dense(4, softmax)
    - Optimizer: Adam
    """
    base = InceptionV3(weights='imagenet', include_top=False,
                       input_shape=input_shape)

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=output, name='InceptionV3_FineTuned')

    for layer in base.layers[:-20]:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True

    model.compile(
        optimizer=Adam(learning_rate=cfg.LR_ADAM),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def build_feature_extractor(model):
    """
    Remove classification head, return feature embedding model.
    Paper Section 3.3.4:
    - Xception:    2048-dim features
    - DenseNet121: 1024-dim features
    - InceptionV3: 2048-dim features
    - Concatenated: 5120-dim
    """
    # Remove the last Dense (softmax) layer → take penultimate Dense output
    feature_layer = model.layers[-3]  # Dense(relu) before final dropout+softmax
    extractor = Model(inputs=model.input,
                      outputs=feature_layer.output,
                      name=f'{model.name}_extractor')
    return extractor


def build_combined_extractor(xception_extractor, densenet_extractor, inception_extractor,
                               input_shape):
    """
    Combined feature extractor (paper Figure 12):
    Concatenates Xception(2048) + DenseNet121(1024) + InceptionV3(2048) = 5120 features
    """
    inp = layers.Input(shape=input_shape, name='combined_input')

    feat_xception  = xception_extractor(inp)     # (None, 2048)
    feat_densenet  = densenet_extractor(inp)      # (None, 1024)
    feat_inception = inception_extractor(inp)     # (None, 2048)

    combined = layers.Concatenate(name='feature_concat')(
        [feat_inception, feat_densenet, feat_xception]
    )  # (None, 5120)

    combined_model = Model(inputs=inp, outputs=combined,
                           name='CombinedFeatureExtractor')
    return combined_model


# Build all three models
print("🔨 Building Xception model...")
xception_model   = build_xception(cfg.IMG_SHAPE, cfg.NUM_CLASSES)
print(f"   Parameters: {xception_model.count_params():,}")

print("🔨 Building DenseNet121 model...")
densenet_model   = build_densenet121(cfg.IMG_SHAPE, cfg.NUM_CLASSES)
print(f"   Parameters: {densenet_model.count_params():,}")

print("🔨 Building InceptionV3 model...")
inception_model  = build_inceptionv3(cfg.IMG_SHAPE, cfg.NUM_CLASSES)
print(f"   Parameters: {inception_model.count_params():,}")

print("\n✅ All three CNN models built successfully")

# =============================================================================
# SECTION 9: TRAINING DATA GENERATOR
# =============================================================================

class BrainTumorDataGenerator(keras.utils.Sequence):
    """
    Efficient Keras data generator for brain tumor dataset.
    Applies preprocessing and augmentation on-the-fly.
    """
    def __init__(self, paths, labels, class_names, config,
                 preprocess_method='CLAHE', augment=False, batch_size=32):
        self.paths      = paths
        self.labels     = labels
        self.class_names = class_names
        self.cfg        = config
        self.method     = preprocess_method
        self.augment    = augment
        self.batch_size = batch_size
        self.preprocessor = MRIPreprocessor(config)
        self.augmentor    = DataAugmentor(config)

        # Label encoding
        self.label_map = {cls: i for i, cls in enumerate(class_names)}
        self.indices   = np.arange(len(self.paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_paths  = [self.paths[i] for i in batch_indices]
        batch_labels = [self.labels[i] for i in batch_indices]

        X = np.zeros((len(batch_paths), self.cfg.IMG_SIZE,
                      self.cfg.IMG_SIZE, 3), dtype=np.float32)
        Y = np.zeros((len(batch_paths), len(self.class_names)), dtype=np.float32)

        for i, (path, label) in enumerate(zip(batch_paths, batch_labels)):
            try:
                img = load_image(path, self.cfg.IMG_SIZE).astype(np.float32) / 255.0
                img = self.preprocessor.preprocess(img, self.method)
                if self.augment:
                    img = self.augmentor.augment_image(img)
                X[i] = np.clip(img, 0, 1)
            except:
                X[i] = np.zeros((self.cfg.IMG_SIZE, self.cfg.IMG_SIZE, 3))

            Y[i, self.label_map.get(label, 0)] = 1.0

        return X, Y

    def on_epoch_end(self):
        if self.augment:
            np.random.shuffle(self.indices)


def get_callbacks(model_name, output_path):
    """Standard training callbacks."""
    return [
        ModelCheckpoint(
            filepath=os.path.join(output_path, 'models', f'{model_name}_best.h5'),
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
        EarlyStopping(
            monitor='val_accuracy', patience=10,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=5,
            min_lr=1e-7, verbose=1
        ),
        CSVLogger(
            os.path.join(output_path, 'logs', f'{model_name}_history.csv')
        )
    ]

print("✅ Data generator and callbacks defined")

# =============================================================================
# SECTION 10: TRAIN CNN MODELS
# =============================================================================

print("\n" + "="*60)
print("SECTION 10: TRAINING CNN MODELS")
print("="*60)

def prepare_train_val_split(paths, labels, val_ratio=0.2, seed=42):
    """Stratified train/validation split."""
    train_p, val_p, train_l, val_l = train_test_split(
        paths, labels, test_size=val_ratio,
        stratify=labels, random_state=seed
    )
    return train_p, val_p, train_l, val_l

def train_cnn_model(model, model_name, train_paths, train_labels,
                     val_paths, val_labels, preprocess_method='CLAHE'):
    """Train a single CNN model with the paper's setup."""
    print(f"\n🚀 Training {model_name} with {preprocess_method} preprocessing...")

    train_gen = BrainTumorDataGenerator(
        train_paths, train_labels, cfg.CLASSES, cfg,
        preprocess_method=preprocess_method,
        augment=True, batch_size=cfg.BATCH_SIZE
    )
    val_gen = BrainTumorDataGenerator(
        val_paths, val_labels, cfg.CLASSES, cfg,
        preprocess_method=preprocess_method,
        augment=False, batch_size=cfg.BATCH_SIZE
    )

    callbacks = get_callbacks(f'{model_name}_{preprocess_method}', cfg.OUTPUT_PATH)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=cfg.EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    val_loss, val_acc = model.evaluate(val_gen, verbose=0)
    print(f"   ✅ {model_name} Val Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")

    return history, val_acc


def plot_training_history(histories, model_names, save_path):
    """Plot training/validation curves (like paper Figure 14)."""
    fig, axes = plt.subplots(len(histories), 2,
                              figsize=(14, 4 * len(histories)))
    if len(histories) == 1:
        axes = [axes]

    for i, (hist, name) in enumerate(zip(histories, model_names)):
        # Accuracy
        axes[i][0].plot(hist.history['accuracy'],     label='Train', color='#2196F3')
        axes[i][0].plot(hist.history['val_accuracy'], label='Val',   color='#FF5722',
                         marker='o', markersize=3)
        best_epoch = np.argmax(hist.history['val_accuracy'])
        best_acc   = hist.history['val_accuracy'][best_epoch]
        axes[i][0].scatter(best_epoch, best_acc, color='red', s=80, zorder=5,
                            label=f'Best epoch={best_epoch}')
        axes[i][0].set_title(f'{name} Training/Validation Accuracy')
        axes[i][0].set_xlabel('Epoch')
        axes[i][0].set_ylabel('Accuracy')
        axes[i][0].legend()
        axes[i][0].grid(alpha=0.3)

        # Loss
        axes[i][1].plot(hist.history['loss'],     label='Train', color='#2196F3')
        axes[i][1].plot(hist.history['val_loss'], label='Val',   color='#FF5722')
        axes[i][1].set_title(f'{name} Training/Validation Loss')
        axes[i][1].set_xlabel('Epoch')
        axes[i][1].set_ylabel('Loss')
        axes[i][1].legend()
        axes[i][1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   ✅ Training curves saved")


# Prepare data split
print("\n📊 Preparing train/validation split...")
train_paths, val_paths, train_labels, val_labels = prepare_train_val_split(
    d1_paths, d1_labels
)
print(f"   Train: {len(train_paths)} | Val: {len(val_paths)}")

# Train all three models with CLAHE (paper's best preprocessing)
histories    = []
val_accs     = {}
PREPROCESS   = 'CLAHE'  # Paper winner

for model, name in [(xception_model,  'Xception'),
                     (densenet_model,  'DenseNet121'),
                     (inception_model, 'InceptionV3')]:
    hist, acc = train_cnn_model(
        model, name, train_paths, train_labels,
        val_paths, val_labels, preprocess_method=PREPROCESS
    )
    histories.append(hist)
    val_accs[name] = acc

    # Save model weights
    model.save(os.path.join(cfg.OUTPUT_PATH, 'models', f'{name}_{PREPROCESS}.h5'))
    gc.collect()

# Plot training histories (paper Figure 14 style)
plot_training_history(
    histories,
    ['Xception', 'DenseNet121', 'InceptionV3'],
    os.path.join(cfg.OUTPUT_PATH, 'plots', 'training_history.png')
)

# Compare validation accuracy across architectures
print("\n📊 Validation Accuracy Comparison:")
for name, acc in sorted(val_accs.items(), key=lambda x: -x[1]):
    print(f"   {name}: {acc*100:.2f}%")

# =============================================================================
# SECTION 11: FEATURE EXTRACTION (Paper Section 3.3.4)
# =============================================================================

print("\n" + "="*60)
print("SECTION 11: FEATURE EXTRACTION (5120-dim concatenated)")
print("="*60)

def extract_features_from_model(extractor, paths, labels, preprocess_method='CLAHE',
                                  batch_size=16, desc='Extracting'):
    """
    Extract features from a list of image paths using a model extractor.
    Returns: features array (N, feature_dim), labels list
    """
    prep = MRIPreprocessor(cfg)
    all_features = []
    all_labels   = []

    for i in tqdm(range(0, len(paths), batch_size), desc=desc):
        batch_paths  = paths[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        batch_imgs = []
        valid_labels = []
        for path, lbl in zip(batch_paths, batch_labels):
            try:
                img = load_image(path, cfg.IMG_SIZE).astype(np.float32) / 255.0
                img = prep.preprocess(img, preprocess_method)
                batch_imgs.append(np.clip(img, 0, 1))
                valid_labels.append(lbl)
            except:
                pass

        if len(batch_imgs) == 0:
            continue

        batch_arr  = np.array(batch_imgs, dtype=np.float32)
        batch_feat = extractor.predict(batch_arr, verbose=0)
        all_features.append(batch_feat)
        all_labels.extend(valid_labels)

    return np.vstack(all_features), all_labels


# Build feature extractors from trained models
print("🔧 Building feature extractors from trained models...")
xception_extractor  = build_feature_extractor(xception_model)
densenet_extractor  = build_feature_extractor(densenet_model)
inception_extractor = build_feature_extractor(inception_model)

print(f"   Xception extractor output:  {xception_extractor.output_shape}")
print(f"   DenseNet extractor output:  {densenet_extractor.output_shape}")
print(f"   Inception extractor output: {inception_extractor.output_shape}")

# Extract features for all images
ALL_PATHS  = d1_paths
ALL_LABELS = d1_labels

print(f"\n📡 Extracting Xception features ({len(ALL_PATHS)} images)...")
xception_features, labels_x = extract_features_from_model(
    xception_extractor, ALL_PATHS, ALL_LABELS,
    preprocess_method=PREPROCESS, desc='Xception'
)

print(f"📡 Extracting DenseNet121 features...")
densenet_features, labels_d = extract_features_from_model(
    densenet_extractor, ALL_PATHS, ALL_LABELS,
    preprocess_method=PREPROCESS, desc='DenseNet'
)

print(f"📡 Extracting InceptionV3 features...")
inception_features, labels_i = extract_features_from_model(
    inception_extractor, ALL_PATHS, ALL_LABELS,
    preprocess_method=PREPROCESS, desc='Inception'
)

# Concatenate features: InceptionV3(2048) + DenseNet121(1024) + Xception(2048) = 5120
print("\n🔗 Concatenating features (5120-dim)...")
min_samples = min(len(xception_features), len(densenet_features), len(inception_features))
combined_features = np.concatenate([
    inception_features[:min_samples],   # 2048
    densenet_features[:min_samples],    # 1024
    xception_features[:min_samples]     # 2048
], axis=1)
combined_labels = labels_i[:min_samples]

print(f"   Combined features shape: {combined_features.shape}")
print(f"   Feature dim: {combined_features.shape[1]} (expected 5120)")

# Save features
np.save(os.path.join(cfg.OUTPUT_PATH, 'features', 'combined_features.npy'),
        combined_features)
np.save(os.path.join(cfg.OUTPUT_PATH, 'features', 'combined_labels.npy'),
        np.array(combined_labels))
print("   ✅ Features saved to disk")

# =============================================================================
# SECTION 12: FEATURE SELECTION (Paper Section 3.4)
# Z-Score → ANOVA F-test → L1 Regularization → SGDClassifier grid search
# =============================================================================

print("\n" + "="*60)
print("SECTION 12: FEATURE SELECTION")
print(f"  5120-dim → ANOVA(k={cfg.ANOVA_K}) → L1max({cfg.L1_MAX_FEAT})")
print("="*60)

# Label encoding
label_encoder = LabelEncoder()
label_encoder.fit(cfg.CLASSES)
y_encoded = label_encoder.transform(combined_labels)

# Train/test split for ML models (80/20)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    combined_features, y_encoded,
    test_size=0.2, stratify=y_encoded, random_state=SEED
)
print(f"   Train: {X_train_raw.shape[0]} | Test: {X_test_raw.shape[0]}")


class FeatureSelector:
    """
    Implements paper's 3-stage feature selection (Section 3.4):
    1. Z-Score Normalization (eq. 22)
    2. ANOVA F-test SelectKBest (eq. 23)
    3. L1 Regularization via SGDClassifier (eq. 24-25)
    """

    def __init__(self, anova_k=2000, l1_max_features=1200):
        self.anova_k         = anova_k
        self.l1_max_features = l1_max_features
        self.scaler          = StandardScaler()    # Z-score
        self.anova_selector  = None
        self.l1_mask         = None
        self.is_fitted       = False

    def fit(self, X, y):
        print(f"\n   Step 1: Z-Score normalization (eq. 22)...")
        X_scaled = self.scaler.fit_transform(X)
        print(f"          Input shape: {X_scaled.shape}")

        print(f"   Step 2: ANOVA F-test → top {self.anova_k} features (eq. 23)...")
        self.anova_selector = SelectKBest(f_classif, k=min(self.anova_k, X_scaled.shape[1]))
        X_anova = self.anova_selector.fit_transform(X_scaled, y)
        print(f"          After ANOVA: {X_anova.shape}")

        print(f"   Step 3: L1 regularization → top {self.l1_max_features} features (eq. 24)...")
        # SGDClassifier with L1 penalty (sparse weights)
        sgd_l1 = SGDClassifier(
            loss='log_loss', penalty='l1',
            alpha=1e-4, max_iter=1000,
            random_state=SEED, n_jobs=-1
        )
        sgd_l1.fit(X_anova, y)

        # Get feature importances (non-zero weights = selected)
        if hasattr(sgd_l1, 'coef_'):
            coef_abs = np.abs(sgd_l1.coef_).mean(axis=0)
            top_indices = np.argsort(coef_abs)[::-1][:self.l1_max_features]
            self.l1_mask = np.zeros(X_anova.shape[1], dtype=bool)
            self.l1_mask[top_indices] = True
        else:
            self.l1_mask = np.ones(X_anova.shape[1], dtype=bool)

        X_selected = X_anova[:, self.l1_mask]
        print(f"          After L1:   {X_selected.shape}")
        self.is_fitted = True
        return X_selected

    def transform(self, X):
        assert self.is_fitted, "Must call fit() first"
        X_scaled = self.scaler.transform(X)
        X_anova  = self.anova_selector.transform(X_scaled)
        X_sel    = X_anova[:, self.l1_mask]
        return X_sel

    def fit_transform(self, X, y):
        return self.fit(X, y)

    def save(self, path):
        joblib.dump(self, path)

    @staticmethod
    def load(path):
        return joblib.load(path)


# Run feature selection
print("\n🔍 Running Feature Selection Pipeline...")
selector = FeatureSelector(anova_k=cfg.ANOVA_K, l1_max_features=cfg.L1_MAX_FEAT)
X_train_sel = selector.fit_transform(X_train_raw, y_train)
X_test_sel  = selector.transform(X_test_raw)

print(f"\n   Feature reduction: {X_train_raw.shape[1]} → {X_train_sel.shape[1]}")
print(f"   Reduction ratio: {X_train_sel.shape[1]/X_train_raw.shape[1]*100:.1f}%")

selector.save(os.path.join(cfg.OUTPUT_PATH, 'models', 'feature_selector.pkl'))
print("   ✅ Feature selector saved")

# =============================================================================
# SECTION 13: MACHINE LEARNING CLASSIFIERS (Paper Section 3.5)
# All 11 classifiers from paper + Grid Search
# =============================================================================

print("\n" + "="*60)
print("SECTION 13: TRAINING 11 ML CLASSIFIERS")
print("="*60)

def get_all_classifiers():
    """
    Returns dict of all 11 classifiers from paper (Section 3.5)
    with their grid search parameter ranges (Table 6).
    """
    classifiers = {

        # 1. K-Nearest Neighbors (eq. 26) - Best: n_neighbors=11, uniform, euclidean
        'K-Nearest Neighbour': {
            'model': KNeighborsClassifier(),
            'params': {
                'n_neighbors': [3, 5, 7, 11, 15],
                'weights':     ['uniform', 'distance'],
                'metric':      ['euclidean', 'manhattan', 'minkowski']
            },
            'best_params': {
                'n_neighbors': 11,
                'weights': 'uniform',
                'metric': 'euclidean'
            }
        },

        # 2. Random Forest (eq. 27) - Best: n_estimators=50, max_depth=None
        'Random Forest': {
            'model': RandomForestClassifier(random_state=SEED, n_jobs=-1),
            'params': {
                'n_estimators':    [50, 100, 200],
                'max_depth':       [10, 20, 30, None],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf':  [1, 2, 4],
                'bootstrap':       [True, False]
            },
            'best_params': {
                'n_estimators': 50,
                'max_depth': None,
                'min_samples_split': 2,
                'min_samples_leaf': 1,
                'bootstrap': True
            }
        },

        # 3. Support Vector Machine (eq. 28) - Best: C=0.1, linear, scale
        'Support Vector Classifier': {
            'model': SVC(probability=True, random_state=SEED),
            'params': {
                'C':      [0.1, 1, 10, 100],
                'kernel': ['linear', 'rbf', 'poly'],
                'gamma':  ['scale', 'auto', 0.001, 0.01, 0.1],
                'degree': [3, 4, 5]
            },
            'best_params': {
                'C': 0.1,
                'kernel': 'linear',
                'gamma': 'scale',
                'degree': 3
            }
        },

        # 4. Gradient Boosting - Best: n_estimators=100
        'Gradient Boosting': {
            'model': GradientBoostingClassifier(
                n_estimators=100, loss='log_loss', random_state=SEED
            ),
            'params': None,  # Paper uses defaults
            'best_params': {'n_estimators': 100}
        },

        # 5. Logistic Regression (eq. 29) - Best: C=1, l2, liblinear
        'Logistic Regression': {
            'model': LogisticRegression(random_state=SEED, max_iter=5000, n_jobs=-1),
            'params': {
                'C':       [0.1, 1, 10, 100],
                'penalty': ['l1', 'l2'],
                'solver':  ['liblinear', 'saga'],
                'max_iter': [1000, 5000]
            },
            'best_params': {
                'C': 1,
                'penalty': 'l2',
                'solver': 'liblinear',
                'max_iter': 1000
            }
        },

        # 6. Decision Tree - Best: max_depth=10, min_split=2, gini
        'Decision Tree': {
            'model': DecisionTreeClassifier(random_state=SEED),
            'params': {
                'max_depth':         [5, 10, 15, 20, None],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf':  [1, 2, 4],
                'criterion':         ['gini', 'entropy']
            },
            'best_params': {
                'max_depth': 10,
                'min_samples_split': 2,
                'min_samples_leaf': 1,
                'criterion': 'gini'
            }
        },

        # 7. Naive Bayes (eq. 30)
        'Naive Bayes': {
            'model': GaussianNB(),
            'params': None,
            'best_params': {}
        },

        # 8. Multilayer Perceptron (eq. 31) - Best: (128,64), relu, adam, 0.0001
        'MultiLayer Perceptron': {
            'model': MLPClassifier(random_state=SEED, max_iter=5000),
            'params': {
                'hidden_layer_sizes': [(64, 32), (128, 64), (256, 128), (512, 256)],
                'activation':         ['relu', 'tanh'],
                'solver':             ['adam', 'sgd'],
                'alpha':              [0.0001, 0.001, 0.01],
                'learning_rate':      ['constant', 'adaptive', 'invscaling'],
                'max_iter':           [1000, 5000]
            },
            'best_params': {
                'hidden_layer_sizes': (128, 64),
                'activation': 'relu',
                'solver': 'adam',
                'alpha': 0.0001,
                'learning_rate': 'constant',
                'max_iter': 5000
            }
        },

        # 9. XGBoost - Best: n_estimators=100, lr=0.01, max_depth=3, subsample=0.8
        'XGBoost': {
            'model': xgb.XGBClassifier(
                random_state=SEED, n_jobs=-1,
                eval_metric='mlogloss', verbosity=0
            ),
            'params': {
                'n_estimators':  [100, 200],
                'learning_rate': [0.01, 0.1, 0.2],
                'max_depth':     [3, 5, 7],
                'subsample':     [0.8, 1.0]
            },
            'best_params': {
                'n_estimators': 100,
                'learning_rate': 0.01,
                'max_depth': 3,
                'subsample': 0.8
            }
        },

        # 10. LightGBM - Best: n_estimators=100, lr=0.1, max_depth=-1, num_leaves=31
        'Light Gradient Boost': {
            'model': lgb.LGBMClassifier(
                random_state=SEED, n_jobs=-1, verbose=-1
            ),
            'params': {
                'n_estimators':  [50, 100, 200],
                'learning_rate': [0.01, 0.1, 0.2],
                'max_depth':     [3, 5, 7, -1],
                'subsample':     [0.8, 1.0],
                'num_leaves':    [31, 50, 100]
            },
            'best_params': {
                'n_estimators': 100,
                'learning_rate': 0.1,
                'max_depth': -1,
                'subsample': 1.0,
                'num_leaves': 31
            }
        },

        # 11. Extra Trees - Best: n_estimators=100, max_depth=None, gini
        'ExtraTrees': {
            'model': ExtraTreesClassifier(random_state=SEED, n_jobs=-1),
            'params': {
                'n_estimators':    [50, 100, 200],
                'max_depth':       [5, 10, 15, None],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf':  [1, 2, 4],
                'criterion':       ['gini', 'entropy']
            },
            'best_params': {
                'n_estimators': 100,
                'max_depth': None,
                'min_samples_split': 2,
                'min_samples_leaf': 1,
                'criterion': 'gini'
            }
        }
    }
    return classifiers


def train_and_evaluate_classifier(name, clf_config, X_train, y_train,
                                   X_test, y_test, use_best_params=True,
                                   use_grid_search=False, cv_folds=3):
    """
    Train and evaluate a single classifier.
    Returns performance metrics dict.
    """
    model = clf_config['model']
    best_params = clf_config.get('best_params', {})

    if use_best_params and best_params:
        # Use paper's reported best params directly (fast)
        try:
            model.set_params(**best_params)
        except Exception as e:
            pass  # Some params may not apply

    elif use_grid_search and clf_config.get('params'):
        # Full grid search (slow but thorough)
        print(f"     Running {cv_folds}-fold GridSearchCV...")
        grid = GridSearchCV(
            model, clf_config['params'],
            cv=StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=SEED),
            scoring='accuracy', n_jobs=-1, verbose=0
        )
        grid.fit(X_train, y_train)
        model = grid.best_estimator_
        print(f"     Best params: {grid.best_params_}")
    
    # Train
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    # Predict
    y_pred = model.predict(X_test)

    # Metrics (paper reports Accuracy, Precision, Recall, F1)
    acc   = accuracy_score(y_test, y_pred)
    prec  = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm    = confusion_matrix(y_test, y_pred)

    return {
        'name':        name,
        'model':       model,
        'accuracy':    acc,
        'precision':   prec,
        'recall':      rec,
        'f1':          f1,
        'confusion_matrix': cm,
        'y_pred':      y_pred,
        'train_time':  train_time
    }


# Train all 11 classifiers
print("🚀 Training all 11 ML classifiers (using paper's best params)...\n")
classifiers_config = get_all_classifiers()
results = {}

for name, clf_config in classifiers_config.items():
    print(f"  Training: {name}...")
    try:
        result = train_and_evaluate_classifier(
            name, clf_config, X_train_sel, y_train, X_test_sel, y_test,
            use_best_params=True, use_grid_search=False
        )
        results[name] = result
        print(f"    ✅ Accuracy: {result['accuracy']*100:.2f}% | "
              f"F1: {result['f1']*100:.2f}% | "
              f"Time: {result['train_time']:.1f}s")

        # Save model
        joblib.dump(result['model'],
                    os.path.join(cfg.OUTPUT_PATH, 'models',
                                 f"{name.replace(' ', '_')}_{PREPROCESS}.pkl"))
    except Exception as e:
        print(f"    ❌ Failed: {e}")

# =============================================================================
# SECTION 14: RESULTS ANALYSIS & VISUALIZATION (Paper Tables 7 & 8)
# =============================================================================

print("\n" + "="*60)
print("SECTION 14: RESULTS ANALYSIS")
print("="*60)

# Build results DataFrame (like paper Table 7)
results_rows = []
for name, r in results.items():
    results_rows.append({
        'Model':              name,
        'Normalization':      PREPROCESS,
        'Accuracy (%)':       round(r['accuracy'] * 100, 2),
        'Precision (%)':      round(r['precision'] * 100, 2),
        'Recall (%)':         round(r['recall'] * 100, 2),
        'F1 Score (%)':       round(r['f1'] * 100, 2),
        'Train Time (s)':     round(r['train_time'], 2)
    })

results_df = pd.DataFrame(results_rows).sort_values('Accuracy (%)', ascending=False)
print("\n📊 Performance Comparison (Table 7 style):")
print(results_df.to_string(index=False))

# Save results
results_df.to_csv(os.path.join(cfg.OUTPUT_PATH, 'results', f'classifier_results_{PREPROCESS}.csv'),
                   index=False)
print(f"\n✅ Results saved to CSV")


def plot_classifier_comparison(results_df, save_path):
    """Bar chart comparison of all classifiers (like paper discussion)."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    models = results_df['Model'].tolist()
    accs   = results_df['Accuracy (%)'].tolist()
    f1s    = results_df['F1 Score (%)'].tolist()
    colors = plt.cm.RdYlGn(np.linspace(0.4, 0.9, len(models)))

    # Accuracy bar chart
    bars = axes[0].barh(models, accs, color=colors)
    axes[0].set_xlabel('Accuracy (%)')
    axes[0].set_title(f'Accuracy Comparison\n({PREPROCESS} Preprocessing)', fontweight='bold')
    axes[0].set_xlim([min(accs) - 1, 100])
    for bar, val in zip(bars, accs):
        axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{val:.2f}%', va='center', fontsize=8)
    axes[0].grid(axis='x', alpha=0.3)

    # F1 Score bar chart
    bars2 = axes[1].barh(models, f1s, color=colors)
    axes[1].set_xlabel('F1 Score (%)')
    axes[1].set_title(f'F1 Score Comparison\n({PREPROCESS} Preprocessing)', fontweight='bold')
    axes[1].set_xlim([min(f1s) - 1, 100])
    for bar, val in zip(bars2, f1s):
        axes[1].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{val:.2f}%', va='center', fontsize=8)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_classifier_comparison(
    results_df,
    os.path.join(cfg.OUTPUT_PATH, 'plots', 'classifier_comparison.png')
)


def plot_confusion_matrix(cm, class_names, title, save_path):
    """Plot confusion matrix (like paper Figure 16)."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)

    thresh = cm.max() / 2
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center', fontsize=11,
                    color='white' if cm[i, j] > thresh else 'black',
                    fontweight='bold')

    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Find best model (MLP expected to win per paper)
best_model_name = results_df.iloc[0]['Model']
best_result     = results[best_model_name]
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Accuracy:  {best_result['accuracy']*100:.2f}%")
print(f"   Precision: {best_result['precision']*100:.2f}%")
print(f"   Recall:    {best_result['recall']*100:.2f}%")
print(f"   F1 Score:  {best_result['f1']*100:.2f}%")

# Plot confusion matrix for best model (paper Figure 16)
plot_confusion_matrix(
    best_result['confusion_matrix'],
    cfg.CLASSES,
    f'Confusion Matrix: {best_model_name}\nwith {PREPROCESS} Normalization (Dataset 1)',
    os.path.join(cfg.OUTPUT_PATH, 'plots', f'confusion_matrix_{best_model_name}.png')
)

# Print classification report
print(f"\n📋 Classification Report - {best_model_name}:")
print(classification_report(
    y_test, best_result['y_pred'],
    target_names=label_encoder.classes_
))

# =============================================================================
# SECTION 15: PREPROCESSING METHOD COMPARISON (Paper Tables 7 & 8 full)
# =============================================================================

print("\n" + "="*60)
print("SECTION 15: COMPARING ALL 5 PREPROCESSING METHODS")
print("="*60)

PREPROCESSING_METHODS = ['CLAHE', 'Nyul', 'WhiteStripe', 'N4', 'Template']
preprocess_comparison_results = {}

for method in PREPROCESSING_METHODS:
    print(f"\n📊 Evaluating preprocessing: {method}")
    try:
        # Extract features with this preprocessing
        x_feat, lbl_x = extract_features_from_model(
            xception_extractor, ALL_PATHS, ALL_LABELS,
            preprocess_method=method, desc=f'Xception[{method}]'
        )
        d_feat, lbl_d = extract_features_from_model(
            densenet_extractor, ALL_PATHS, ALL_LABELS,
            preprocess_method=method, desc=f'DenseNet[{method}]'
        )
        i_feat, lbl_i = extract_features_from_model(
            inception_extractor, ALL_PATHS, ALL_LABELS,
            preprocess_method=method, desc=f'Inception[{method}]'
        )

        n = min(len(x_feat), len(d_feat), len(i_feat))
        feats = np.concatenate([i_feat[:n], d_feat[:n], x_feat[:n]], axis=1)
        lbls  = lbl_i[:n]
        y_enc = label_encoder.transform(lbls)

        Xtr, Xte, ytr, yte = train_test_split(
            feats, y_enc, test_size=0.2,
            stratify=y_enc, random_state=SEED
        )

        # Feature selection
        sel = FeatureSelector(cfg.ANOVA_K, cfg.L1_MAX_FEAT)
        Xtr_s = sel.fit_transform(Xtr, ytr)
        Xte_s = sel.transform(Xte)

        # Train MLP (paper's best classifier) only for speed
        mlp_cfg = get_all_classifiers()['MultiLayer Perceptron']
        r = train_and_evaluate_classifier(
            'MultiLayer Perceptron', mlp_cfg, Xtr_s, ytr, Xte_s, yte,
            use_best_params=True
        )

        preprocess_comparison_results[method] = {
            'MLP_Accuracy':  r['accuracy'] * 100,
            'MLP_Precision': r['precision'] * 100,
            'MLP_Recall':    r['recall'] * 100,
            'MLP_F1':        r['f1'] * 100
        }
        print(f"   MLP Accuracy with {method}: {r['accuracy']*100:.2f}%")
        gc.collect()

    except Exception as e:
        print(f"   ❌ {method} failed: {e}")
        preprocess_comparison_results[method] = {
            'MLP_Accuracy': 0, 'MLP_Precision': 0, 'MLP_Recall': 0, 'MLP_F1': 0
        }

# Plot preprocessing comparison (paper Figure discussion)
preproc_df = pd.DataFrame(preprocess_comparison_results).T
print("\n📊 Preprocessing Method Comparison (MLP Classifier):")
print(preproc_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(preproc_df))
ax.bar(x, preproc_df['MLP_Accuracy'],
       color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'],
       edgecolor='black', width=0.6)
ax.set_xticks(x)
ax.set_xticklabels(preproc_df.index, fontsize=11)
ax.set_ylabel('MLP Accuracy (%)')
ax.set_title('Preprocessing Method Comparison\n(MLP Classifier, Dataset 1)',
              fontweight='bold')
ax.set_ylim([preproc_df['MLP_Accuracy'].min() - 1, 100])
for i, val in enumerate(preproc_df['MLP_Accuracy']):
    ax.text(i, val + 0.1, f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', 'preprocessing_comparison_accuracy.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# =============================================================================
# SECTION 16: COMPARISON WITH SOTA (Paper Table 9)
# =============================================================================

print("\n" + "="*60)
print("SECTION 16: COMPARISON WITH STATE-OF-THE-ART")
print("="*60)

sota_data = {
    'Method': [
        'Yazdan et al. (Multiscale CNN)',
        'Malla et al. (VGGNet DCNN)',
        'Islam et al. (PCA+TK-means)',
        'Rehman et al. (GoogLeNet+AlexNet+VGG)',
        'Swati et al. (Block-wise TL)',
        'Deepak et al. (GoogLeNet TL)',
        'Latif et al. (CNN+SVM)',
        'Amin et al. (SVM+handcrafted)',
        'Rinesh et al. (KNN+K-means)',
        'Nayak et al. (LBP+GWT)',
        'Anaya-Isaza et al. (PCA)',
        f'Proposed (Xception+DenseNet+InceptionV3+MLP)'
    ],
    'Dataset': [
        'Kaggle (3264)', 'Figshare', '40 MRI', 'Figshare',
        'Brain MRI', 'Public (1426)', 'BraTS', 'Multi-center',
        '250 samples', '2 datasets', '3264 MR', 'Dataset 1 & 2'
    ],
    'Accuracy (%)': [
        91.20, 98.93, 95.00, 98.69,
        94.82, 98.00, 96.19, 97.10,
        96.47, 97.20, 92.13,
        round(best_result['accuracy'] * 100, 2)
    ]
}

sota_df = pd.DataFrame(sota_data)
sota_df = sota_df.sort_values('Accuracy (%)')

print(sota_df.to_string(index=False))
sota_df.to_csv(os.path.join(cfg.OUTPUT_PATH, 'results', 'sota_comparison.csv'), index=False)

# Plot SOTA comparison (Table 9 visualization)
fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#B0B0B0'] * (len(sota_df) - 1) + ['#FF6B35']
bars = ax.barh(sota_df['Method'], sota_df['Accuracy (%)'],
               color=colors, edgecolor='black', height=0.7)
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Comparison with State-of-the-Art Methods\n(Brain Tumor Classification)',
              fontsize=13, fontweight='bold')
ax.set_xlim([88, 101])
for bar, val in zip(bars, sota_df['Accuracy (%)']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=8,
            fontweight='bold' if val >= 99 else 'normal')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', 'sota_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# =============================================================================
# SECTION 17: BTdiagAI INFERENCE FUNCTION (Web App Logic)
# =============================================================================

print("\n" + "="*60)
print("SECTION 17: BTdiagAI INFERENCE PIPELINE")
print("="*60)

class BTdiagAI:
    """
    Complete inference pipeline — equivalent to BTdiagAI web app.
    Replicates https://brain-tumor-detect-ai.streamlit.app/
    Input:  MRI image (path or numpy array)
    Output: tumor class + confidence scores
    """

    def __init__(self, xception_ext, densenet_ext, inception_ext,
                 feature_selector, classifier, label_encoder,
                 preprocess_method='CLAHE', config=None):
        self.xception_ext   = xception_ext
        self.densenet_ext   = densenet_ext
        self.inception_ext  = inception_ext
        self.selector       = feature_selector
        self.classifier     = classifier
        self.label_encoder  = label_encoder
        self.method         = preprocess_method
        self.cfg            = config or Config()
        self.preprocessor   = MRIPreprocessor(self.cfg)

    def predict(self, image_input):
        """
        Classify brain MRI image.
        image_input: file path (str) or numpy array (H, W, 3)
        Returns: {
            'primary_class': str,
            'confidence': float,
            'all_scores': dict,
            'confidence_percent': str
        }
        """
        # Load image
        if isinstance(image_input, str):
            img = load_image(image_input, self.cfg.IMG_SIZE).astype(np.float32) / 255.0
        else:
            img = image_input.astype(np.float32)
            if img.max() > 1.0:
                img = img / 255.0
            img = cv2.resize(img, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))

        # Preprocess
        img_proc = self.preprocessor.preprocess(img, self.method)
        img_batch = np.expand_dims(np.clip(img_proc, 0, 1), 0)

        # Extract features
        feat_x = self.xception_ext.predict(img_batch,  verbose=0)  # (1, 2048)
        feat_d = self.densenet_ext.predict(img_batch,  verbose=0)  # (1, 1024)
        feat_i = self.inception_ext.predict(img_batch, verbose=0)  # (1, 2048)

        # Concatenate
        combined = np.concatenate([feat_i, feat_d, feat_x], axis=1)  # (1, 5120)

        # Feature selection
        selected = self.selector.transform(combined)

        # Classify
        pred_class_idx = self.classifier.predict(selected)[0]
        pred_class     = self.label_encoder.inverse_transform([pred_class_idx])[0]

        # Confidence scores
        if hasattr(self.classifier, 'predict_proba'):
            proba       = self.classifier.predict_proba(selected)[0]
            all_classes = self.label_encoder.inverse_transform(
                range(len(self.label_encoder.classes_))
            )
            scores = {cls: float(p) for cls, p in zip(all_classes, proba)}
            confidence = scores[pred_class]
        else:
            scores     = {cls: (1.0 if cls == pred_class else 0.0)
                         for cls in self.label_encoder.classes_}
            confidence = 1.0

        return {
            'primary_class':      pred_class,
            'confidence':         confidence,
            'confidence_percent': f'{confidence * 100:.1f}%',
            'all_scores':         scores,
            'is_tumor':           pred_class != 'notumor'
        }

    def display_result(self, image_input, result):
        """Display classification result (like paper Figure 17)."""
        if isinstance(image_input, str):
            img = load_image(image_input, self.cfg.IMG_SIZE)
        else:
            img = (image_input * 255).astype(np.uint8) if image_input.max() <= 1 else image_input

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle('BTdiagAI: Brain Tumor Detection from MRI Scans',
                      fontsize=14, fontweight='bold')

        # MRI scan
        axes[0].imshow(img)
        axes[0].set_title('MRI Scan Preview', fontsize=11)
        axes[0].axis('off')

        # Confidence scores (horizontal bar like paper Figure 17)
        classes = list(result['all_scores'].keys())
        scores  = list(result['all_scores'].values())
        colors  = ['#FF6B35' if c == result['primary_class'] else '#B0B0B0'
                   for c in classes]

        bars = axes[1].barh(classes, scores, color=colors, edgecolor='black')
        axes[1].set_xlim([0, 1.05])
        axes[1].set_xlabel('Confidence Score')
        axes[1].set_title('Classification Confidence', fontsize=11)

        for bar, score in zip(bars, scores):
            axes[1].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                         f'{score:.2f}', va='center', fontsize=9)
        axes[1].grid(axis='x', alpha=0.3)

        # Clinical diagnosis box
        cls_color = '#FF4444' if result['is_tumor'] else '#44BB44'
        fig.text(0.5, 0.02,
                 f"🩺 Primary Classification: {result['primary_class'].upper()}   "
                 f"| Confidence: {result['confidence_percent']}",
                 ha='center', fontsize=13, fontweight='bold', color=cls_color,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow',
                           edgecolor=cls_color, alpha=0.8))

        plt.tight_layout(rect=[0, 0.08, 1, 1])
        plt.savefig(os.path.join(cfg.OUTPUT_PATH, 'plots', 'btdiagai_demo.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()


# Initialize BTdiagAI with best model (MLP)
best_clf_name = results_df.iloc[0]['Model']
best_clf      = results[best_clf_name]['model']

btdiag = BTdiagAI(
    xception_extractor, densenet_extractor, inception_extractor,
    selector, best_clf, label_encoder,
    preprocess_method=PREPROCESS, config=cfg
)

# Demo prediction on a test image
print(f"\n🤖 BTdiagAI Demo Prediction (using {best_clf_name}):")
test_idx  = 0
test_path = val_paths[test_idx]
true_lbl  = val_labels[test_idx]

pred_result = btdiag.predict(test_path)
print(f"   Image:           {os.path.basename(test_path)}")
print(f"   True Label:      {true_lbl}")
print(f"   Predicted:       {pred_result['primary_class']}")
print(f"   Confidence:      {pred_result['confidence_percent']}")
print(f"   Is Tumor:        {pred_result['is_tumor']}")
print(f"   All Scores:      {pred_result['all_scores']}")

btdiag.display_result(test_path, pred_result)

# =============================================================================
# SECTION 18: FINAL SUMMARY REPORT
# =============================================================================

print("\n" + "="*60)
print("SECTION 18: FINAL SUMMARY REPORT")
print("="*60)

print("""
╔══════════════════════════════════════════════════════════╗
║           BTdiagAI Implementation Summary                ║
║     Ahmmed et al., Healthcare Technology Letters 2026    ║
╠══════════════════════════════════════════════════════════╣
║  Paper Architecture:                                     ║
║  • Preprocessing:  CLAHE (winner), Nyul, WhiteStripe,   ║
║                    N4 Bias, Template Registration        ║
║  • Feature Extractor: Xception(2048) + DenseNet121(1024) ║
║                       + InceptionV3(2048) = 5120 dim    ║
║  • Feature Selection: ANOVA(k=2000) → L1max(1200)       ║
║  • Classifiers:    11 ML models (MLP best)               ║
╠══════════════════════════════════════════════════════════╣
""")

print(f"  CNN Validation Accuracies (CLAHE preprocessing):")
for name, acc in sorted(val_accs.items(), key=lambda x: -x[1]):
    print(f"    {name:20s}: {acc*100:.2f}%")

print(f"\n  ML Classifier Results (Dataset 1, CLAHE):")
for _, row in results_df.head(5).iterrows():
    print(f"    {row['Model']:30s}: {row['Accuracy (%)']:.2f}%")

print(f"\n  Best Configuration:")
print(f"    Preprocessing:  {PREPROCESS}")
print(f"    Best Classifier: {best_clf_name}")
print(f"    Accuracy:       {results[best_clf_name]['accuracy']*100:.2f}%")
print(f"    F1 Score:       {results[best_clf_name]['f1']*100:.2f}%")

print(f"""
╠══════════════════════════════════════════════════════════╣
║  Paper Reported Results:                                 ║
║    Dataset 1 (CLAHE + MLP): 99.80%                      ║
║    Dataset 2 (CLAHE + MLP): 99.61%                      ║
╠══════════════════════════════════════════════════════════╣
║  Saved Files:                                            ║
║    /kaggle/working/models/   - CNN & ML models          ║
║    /kaggle/working/features/ - Extracted features       ║
║    /kaggle/working/results/  - CSV result tables        ║
║    /kaggle/working/plots/    - All visualizations       ║
╚══════════════════════════════════════════════════════════╝
""")

print("✅ BTdiagAI implementation complete!")
print("🌐 Web app: https://brain-tumor-detect-ai.streamlit.app/")